In [2]:
import ast
import subprocess
import pandas as pd

for tag in ("min", "max"):
    isolado = pd.read_csv(f"dados/consumo-ativo/media_sequencial_{tag}.csv").set_index("método")["Média"]

    paralelo = pd.read_csv(f"dados/consumo-ativo/media_paralelo_{tag}.csv")
    paralelo["par"] = paralelo["par"].apply(ast.literal_eval)   # string -> tupla real

    linhas = []
    for _, row in paralelo.iterrows():
        m1, m2 = row["par"]
        a1, a2, a_s = isolado[m1], isolado[m2], row["Média"]
        saida = subprocess.run(
            ["venv/bin/python", "scripts/calculo-baseline.py", str(a_s), str(a1), str(a2)],
            capture_output=True, text=True, check=True,
        )
        b1, b2 = (float(x) for x in saida.stdout.split())
        linhas.append({"cenario": tag, "P1": m1, "P2": m2,
                        "A_P1": a1, "A_P2": a2, "A_S": a_s,
                        "baseline_P1": b1, "baseline_P2": b2})

    pd.DataFrame(linhas).to_csv(f"dados/baseline/baseline-R{tag}.csv", index=False)
    print(f"R{tag}: {len(linhas)} pares processados")

Rmin: 55 pares processados
Rmax: 55 pares processados
